# QMC energy benchmarks for the 3D toric-code NQS

Independent, non-variational ground-state energies for
$H=-\sum_v A_v-\sum_p B_p-h_x\sum_i\sigma^x_i-h_z\sum_i\sigma^z_i$
on the open $L^3$ cubic lattice, at the mixed-field point where the NQS data lives
and where no exact 4th-order series exists.

**Engines** (both CPU-only — do *not* waste a GPU runtime on this notebook):
- **ParaToric 1.0** (Linsel & Pollet, SciPost Phys. Codebases **75** (2026); Wu *et al.*, PRB **85**, 195104 (2012)) — continuous-time worldline QMC purpose-built for the toric code in parallel fields; primary engine, all `L`.
- **PMRQMC** (Barash, Babakhani, Hen, PRR **6**, 013281 (2024)) — permutation-matrix-representation QMC over arbitrary Pauli strings; independent-algorithm cross-check at `L=4`.

Both are finite-$T$ methods: $\langle H\rangle\to E_0$ **from above, monotonically**, with thermal
error $\lesssim 2N_{\rm stab}e^{-2\beta}$ ($<10^{-7}$ at $\beta=12$) — checked by $\beta$-doubling.

**Validation ladder** (all targets exact, computed offline in `analysis/scripts/exact_benchmarks.py` /
`analysis/scripts/export_pmrqmc.py`): (a) $h=0$ anchors $E_0=-(L^3+3L(L-1)^2)$, which also certifies that
ParaToric's "smooth open" cubic geometry is ours; (b) $L=2$ OBC ED at the production point;
(c) pure-$h_z$ point vs the exact 4th-order series; (d) $\beta$-doubling.
PMRQMC itself is already validated locally on this ladder: at $L=2$, $\beta=10$ it gave
$E=-14.18585(1591)$ vs ED $-14.1864713$ ($z=0.04$), with $\langle\mathrm{sgn}\rangle=1$ and
$\mathrm{Var}(H)\approx0$.

Resume-safe: every (engine, L) result is a JSON in `OUTDIR/raw/` and finished points are skipped —
mount Drive and re-run all cells after a disconnect. Set `SMOKE=True` first to test the pipeline
end-to-end in minutes.

In [ ]:
# ====== 1 · CONFIG — the one cell to edit ======
import os, json, time

SMOKE       = False          # True = ~100x fewer samples (pipeline test, minutes)
MOUNT_DRIVE = True           # persist results across Colab disconnects
LS          = [4, 5, 6, 7]   # production system sizes
HX, HZ      = 0.2, 0.1       # production point (NQS energies exist here for all LS)
BETA        = 12.0           # thermal error ~ 2*Nstab*exp(-2*beta) < 1e-7
BASIS       = "x"            # ParaToric heuristic: 'x' basis when h/J > lambda/mu
N_CHAINS    = max(2, os.cpu_count() or 2)   # independent seeds per point
N_BLOCKS    = 4                             # blocks per chain -> progress prints + running E
N_SAMPLES   = 20_000                        # stored samples per chain (split over N_BLOCKS)
# ~120 update-steps per edge between samples; LESS gives a uniform negative bias
# (validated against exact h=0 anchors L=2..7 locally, 2026-07-30). Thermalization
# scales with beta in the driver; L=6 showed chi2_red~2 -> its therm doubled.
N_BETWEEN   = {2: 2_000, 4: 16_000, 5: 36_000, 6: 64_000, 7: 104_000}
N_THERM     = {2: 30_000, 4: 250_000, 5: 500_000, 6: 1_600_000, 7: 1_300_000}
N_RESAMPLES = 1_000                         # ParaToric bootstrap resamples
OBS         = ["energy", "star_x", "plaquette_z", "sigma_x", "sigma_z"]

RUN_VALIDATION = True
RUN_PMRQMC     = True        # cross-check at L=4 with the independent algorithm
# Calibration (measured, L=2 OBC, beta=10): 275 us/update, sigma(E)=0.016 per 1.1e7
# updates, E = -14.18585(1591) vs ED -14.18647 (z=0.04). PMR cost grows ~beta^2.2 and
# ~linearly with <q> ~ beta*|H_offdiag|, so at L=4 expect ~1-2 ms/update: the settings
# below give sigma(E) ~ 0.03 in roughly 3-8 h. beta=6 thermal offset ~ 2*172*e^-12
# ~ +2e-3 (approaches E0 from above), well below that sigma. Tighter cross-checks
# belong on many-core hardware (NERSC CPU), not Colab.
PMR_BETA, PMR_QMAX   = 6.0, 2_000
PMR_TSTEPS, PMR_STEPS = 3_000_000, 30_000_000
PMR_CHAINS = 2

if SMOKE:
    N_SAMPLES //= 100; N_RESAMPLES = 200
    N_THERM = {k: v // 100 for k, v in N_THERM.items()}
    PMR_TSTEPS //= 100; PMR_STEPS //= 100

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUTDIR = "/content/drive/MyDrive/tc_qmc" if MOUNT_DRIVE else "/content/tc_qmc"
os.makedirs(f"{OUTDIR}/raw", exist_ok=True)

# ---- exact offline references (analysis/scripts/exact_benchmarks.py, verified vs ED/series) ----
REF = {
    "anchor_E0": {2: -14.0, 4: -172.0, 5: -365.0, 6: -666.0, 7: -1099.0},
    "counts": {   # N spins; exact 2nd-order coefficients; exact pure-hz 4th-order c_z4
        2: dict(N=12,  c_z2=3.0,   c_x2=3.0,    c_z4=2.4375),
        4: dict(N=144, c_z2=36.0,  c_x2=25.5,   c_z4=48.0),
        5: dict(N=300, c_z2=75.0,  c_x2=49.5,   c_z4=107.34375),
        6: dict(N=540, c_z2=135.0, c_x2=85.0,   c_z4=201.9375),
        7: dict(N=882, c_z2=220.5, c_x2=134.25, c_z4=339.9375),
    },
    "L2_ED": {"hx": 0.2, "hz": 0.1, "E": -14.1864712786},   # matrix-free ED, this geometry
    # NQS single-point vs.expect(H) at (hx=0.2, hz=0.1): results/phase_hx0.2_energy/
    "NQS": {4: (-173.40859309426926, 0.02221527804780529),
            5: (-367.7097739791383,  0.01990040402456012),
            6: (-670.717691337438,   0.025503726547563284),
            7: (-1105.9966348144494, 0.0912515293957911)},
    # NQS at the pure-hz validation point (hx=0, hz=0.1), L=4: results/phase_hx0.0_energy/
    "NQS_hz0.1_L4": (-172.36039405806656, 0.003582109938215053),
}

def series2(L, hx, hz):
    c = REF["counts"][L]
    return REF["anchor_E0"][L] - c["c_z2"] * hz**2 - c["c_x2"] * hx**2

def series4_hz(L, hz):        # pure-hz only (exact through O(hz^4); trunc ~0.45*N*hz^6)
    c = REF["counts"][L]
    return REF["anchor_E0"][L] - c["c_z2"] * hz**2 - c["c_z4"] * hz**4

print(f"OUTDIR={OUTDIR}  chains={N_CHAINS}  smoke={SMOKE}")
for L in LS:
    print(f"L={L}: series2({HX},{HZ}) = {series2(L, HX, HZ):.4f},  "
          f"NQS = {REF['NQS'][L][0]:.4f} +- {REF['NQS'][L][1]:.4f}")

In [ ]:
# ====== 2 · Toolchain + build ParaToric (idempotent, ~5-10 min first time) ======
import ctypes, glob, re, subprocess, sys

def sh(step, cmd):
    '''Run a build step with its output streamed into the cell (so failures are visible).'''
    print(f"--- {step} ---", flush=True)
    p = subprocess.Popen(["bash", "-c", "set -e\n" + cmd], cwd="/content", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"build step failed: {step}")

sh("micromamba", r'''
export MAMBA_ROOT_PREFIX=/content/mamba_root
mkdir -p $MAMBA_ROOT_PREFIX
[ -x bin/micromamba ] || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest \
  | tar -xj bin/micromamba
bin/micromamba --version''')

sh("toolchain env (gcc>=15, boost, hdf5; ~3 min)", r'''
export MAMBA_ROOT_PREFIX=/content/mamba_root
# ParaToric is tested with GCC 15; GCC 16 breaks its std::println calls -> pin 15.
if ! { [ -x tcqmc/bin/g++ ] && tcqmc/bin/g++ --version | grep -q " 15\."; }; then
  rm -rf tcqmc
  bin/micromamba create -y -q -r $MAMBA_ROOT_PREFIX -p /content/tcqmc -c conda-forge \
    "gcc=15" "gxx=15" "libboost-devel>=1.87" "hdf5>=1.14.3" cmake ninja
fi
tcqmc/bin/g++ --version | head -1''')

sh("python dev headers", r'''
INC=$(python3 -c "import sysconfig; print(sysconfig.get_paths()['include'])")
if [ ! -f "$INC/Python.h" ]; then
  apt-get -qq update && apt-get -qq install -y python3-dev
fi
# Ubuntu multiarch: /usr/include/python3.12/pyconfig.h is a stub that includes
# <x86_64-linux-gnu/python3.12/pyconfig.h>. The system gcc finds that via its builtin
# multiarch path; the conda gcc does NOT search /usr/include/* at all, so (1) make sure
# the real header exists, (2) symlink it under the -isystem dir the build already uses.
if [ ! -f /usr/include/x86_64-linux-gnu/python3.12/pyconfig.h ]; then
  apt-get -qq update && apt-get -qq install -y libpython3.12-dev
fi
ls -l /usr/include/x86_64-linux-gnu/python3.12/pyconfig.h
mkdir -p /usr/include/python3.12/x86_64-linux-gnu
ln -sfn /usr/include/x86_64-linux-gnu/python3.12 \
        /usr/include/python3.12/x86_64-linux-gnu/python3.12
echo "Python.h: $INC"''')

sh("clone ParaToric (recursive: pybind11 submodule)", r'''
[ -f ParaToric/CMakeLists.txt ] || { rm -rf ParaToric; \
  git clone --recursive --depth 1 --shallow-submodules \
    https://github.com/palmbart/ParaToric; }
[ -f ParaToric/external/pybind11/CMakeLists.txt ] || \
  git -C ParaToric submodule update --init --depth 1
# upstream bug: lattice.cpp calls std::println but never includes <print>
python3 - <<'PY'
p = "ParaToric/src/lattice/lattice.cpp"
s = open(p).read()
if "#include <print>" not in s:
    s = s.replace("#include <numeric>", "#include <numeric>\n#include <print>", 1)
    open(p, "w").write(s)
print("lattice.cpp has #include <print>:", "#include <print>" in s)
PY''')

sh("cmake configure + build", r'''
# Rebuild if the extension is missing OR lacks the baked-in RPATH: the conda
# runtime libs (libstdc++ from gcc 15, boost, hdf5) live in /content/tcqmc/lib,
# and an RPATH in the .so itself is the only fix that works in every process
# (parent, forked worker, spawned worker) with no env-var or preload tricks.
SO=$(ls ParaToric/python/paratoric/_paratoric*.so 2>/dev/null || true)
# rebuild until the extension has NO dynamic libstdc++ dependency (manylinux-style
# -static-libstdc++): that is immune to soname preemption by system libs in workers
if [ -z "$SO" ] || ldd "$SO" | grep -q "libstdc++"; then
  rm -f ParaToric/python/paratoric/_paratoric*.so
  rm -rf ParaToric/build            # never reuse a cache from another config
  export PATH=/content/tcqmc/bin:$PATH
  export CC=/content/tcqmc/bin/gcc CXX=/content/tcqmc/bin/g++
  /content/tcqmc/bin/cmake -S ParaToric -B ParaToric/build -G Ninja \
    -DCMAKE_BUILD_TYPE=Release -DPARATORIC_ENABLE_NATIVE_OPT=ON \
    -DPARATORIC_LINK_MPI=OFF -DPARATORIC_BUILD_CLI=OFF -DPARATORIC_BUILD_TESTS=OFF \
    -DPARATORIC_BUILD_PYBIND=ON \
    -DBOOST_ROOT=/content/tcqmc -DHDF5_ROOT=/content/tcqmc \
    -DPython3_EXECUTABLE=$(which python3) \
    -DPYBIND11_FINDPYTHON=NEW -DPython_EXECUTABLE=$(which python3) \
    -DCMAKE_MODULE_LINKER_FLAGS="-static-libstdc++ -static-libgcc -Wl,-rpath,/content/tcqmc/lib -Wl,--disable-new-dtags" \
    -DCMAKE_SHARED_LINKER_FLAGS="-static-libstdc++ -static-libgcc -Wl,-rpath,/content/tcqmc/lib -Wl,--disable-new-dtags"
  /content/tcqmc/bin/cmake --build ParaToric/build -j$(nproc)
fi
ls ParaToric/python/paratoric/_paratoric*.so
echo "--- linkage check (must show NO libstdc++ line) ---"
ldd ParaToric/python/paratoric/_paratoric*.so | grep -E "stdc\+\+|hdf5|boost|not found" || echo "no libstdc++/hdf5/boost dynamic deps unresolved"''')

# Children of the process pools below start fresh (spawn/forkserver): they read
# LD_LIBRARY_PATH at exec time, so exporting it here covers every worker. The
# already-running parent ignores it -> it additionally needs the ctypes preload.
os.environ["LD_LIBRARY_PATH"] = "/content/tcqmc/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

def preload_conda_libs():
    '''Make the conda-forge runtime libs visible to the already-running python.'''
    for name in ("libstdc++.so.6", "libgcc_s.so.1"):
        ctypes.CDLL(f"/content/tcqmc/lib/{name}", mode=ctypes.RTLD_GLOBAL)
    so = glob.glob("/content/ParaToric/python/paratoric/_paratoric*.so")[0]
    for _ in range(3):   # iterate: dependencies of dependencies
        out = subprocess.run(["ldd", so], capture_output=True, text=True).stdout
        missing = re.findall(r"(\S+) => not found", out)
        if not missing:
            break
        for name in missing:
            ctypes.CDLL(f"/content/tcqmc/lib/{name}", mode=ctypes.RTLD_GLOBAL)

preload_conda_libs()
sys.path.insert(0, "/content/ParaToric/python")
import paratoric
# __init__ swallows extension-load failures (version falls back to "0+local"!) --
# only touching the real API proves the .so loaded:
print("extension OK:", paratoric.extended_toric_code.get_sample)

In [ ]:
# ====== 3 · ParaToric driver (one chain per worker process) ======
import numpy as np
from concurrent.futures import ProcessPoolExecutor

def run_chain(job):
    '''One Markov chain at one (L, hx, hz, beta). Returns per-observable stats.'''
    import paratoric
    t0 = time.time()
    series, mean, std, _, _, tau = paratoric.extended_toric_code.get_sample(
        N_samples=job["ns"], N_thermalization=job["nth"], N_between_samples=job["nbs"],
        beta=job["beta"], mu=1.0, h=job["hx"], h_therm=job["hx"], J=1.0,
        lmbda=job["hz"], lmbda_therm=job["hz"], N_resamples=N_RESAMPLES,
        custom_therm=False, observables=OBS, seed=job["seed"], basis=job["basis"],
        lattice_type="cubic", system_size=job["L"], boundaries="open", default_spin=1)
    out = {o: (float(mean[k]), float(std[k]), float(tau[k])) for k, o in enumerate(OBS)}
    out["runtime_s"] = time.time() - t0
    return out

def run_point(L, hx, hz, beta, n_chains, ns=None, basis=None, seed0=1000, quiet=False):
    '''n_chains x N_BLOCKS independent blocks -> inverse-variance-weighted E.

    ParaToric is silent while sampling, so each (chain, block) is a separate
    short run (independent seed; ~0.5% thermalization overhead per block): every
    completion prints a progress line with the running energy estimate.
    '''
    from concurrent.futures import as_completed
    n_blocks = 1 if quiet else N_BLOCKS
    nth = int(N_THERM[L] * max(1.0, beta / 12.0))
    jobs = [dict(L=L, hx=hx, hz=hz, beta=beta, ns=max(1000, (ns or N_SAMPLES) // n_blocks),
                 nbs=N_BETWEEN[L], nth=nth, seed=seed0 + 7 * i,
                 basis=basis or BASIS) for i in range(n_chains * n_blocks)]
    chains, t0 = [], time.time()
    # initializer: forked workers inherit the parent preloads, spawned ones do not
    with ProcessPoolExecutor(max_workers=N_CHAINS, initializer=preload_conda_libs) as ex:
        futs = [ex.submit(run_chain, j) for j in jobs]
        for k, f in enumerate(as_completed(futs), 1):
            chains.append(f.result())
            e = np.array([c["energy"][0] for c in chains])
            se = np.array([c["energy"][1] for c in chains])
            w = 1.0 / se**2
            if not quiet:
                print(f"  [L={L} {k}/{len(jobs)} blocks, {(time.time()-t0)/60:5.1f} min] "
                      f"running E = {np.sum(w*e)/np.sum(w):.6f} "
                      f"+- {np.sqrt(1.0/np.sum(w)):.6f}", flush=True)
    E  = np.array([c["energy"][0] for c in chains])
    sE = np.array([c["energy"][1] for c in chains])
    w = 1.0 / sE**2
    Ebar = float(np.sum(w * E) / np.sum(w))
    err = float(np.sqrt(1.0 / np.sum(w)))
    chi2r = float(np.sum(w * (E - Ebar) ** 2) / max(1, len(E) - 1))
    err *= max(1.0, np.sqrt(chi2r))          # inflate if chains scatter beyond bootstrap
    return dict(L=L, hx=hx, hz=hz, beta=beta, E=Ebar, E_err=err, chi2_red=chi2r,
                chains=chains, n_chains=n_chains)

print("driver ready")

In [ ]:
# ====== 4 · Validation ladder (exact targets; ~minutes) ======
if RUN_VALIDATION:
    ns_val = max(200, N_SAMPLES // 10)
    rows = []
    for L in LS:                                        # (a) h=0 anchors == geometry check
        r = run_point(L, 0.0, 0.0, BETA, 2, ns=ns_val, quiet=True)
        tgt = REF["anchor_E0"][L]
        rows.append((f"anchor L={L}", r["E"], r["E_err"], tgt))
    r = run_point(2, HX, HZ, BETA, 4, ns=ns_val, quiet=True)        # (b) L=2 OBC vs matrix-free ED
    rows.append(("L=2 vs ED", r["E"], r["E_err"], REF["L2_ED"]["E"]))
    r = run_point(4, 0.0, 0.1, BETA, 4, ns=ns_val, basis="z", quiet=True)   # (c) pure-hz vs exact series4
    rows.append(("L=4 hz=0.1 vs series4", r["E"], r["E_err"], series4_hz(4, 0.1)))
    r1 = run_point(4, HX, HZ, BETA, 2, ns=ns_val, quiet=True)       # (d) beta-doubling at the prod point
    r2 = run_point(4, HX, HZ, 2 * BETA, 2, ns=ns_val, quiet=True)
    rows.append(("L=4 beta-doubling", r1["E"] - r2["E"],
                 np.hypot(r1["E_err"], r2["E_err"]), 0.0))

    print(f"{'check':24s} {'E_QMC':>14s} {'err':>9s} {'target':>14s} {'z':>6s}")
    ok = True
    for name, E, err, tgt in rows:
        z = (E - tgt) / err
        ok &= abs(z) < 3
        print(f"{name:24s} {E:14.6f} {err:9.6f} {tgt:14.6f} {z:6.2f} "
              f"{'PASS' if abs(z) < 3 else 'FAIL'}")
    assert ok, "validation failed -- do NOT trust production numbers"
else:
    print("validation skipped")

In [ ]:
# ====== 5 · Production: (hx, hz) point at all L (resume-safe) ======
for L in LS:
    path = f"{OUTDIR}/raw/paratoric_L{L}_hx{HX}_hz{HZ}_beta{BETA}.json"
    if os.path.exists(path):
        print(f"L={L}: exists, skipping ({path})")
        continue
    t0 = time.time()
    r = run_point(L, HX, HZ, BETA, N_CHAINS)
    r["engine"], r["n_samples"], r["obs"] = "paratoric", N_SAMPLES, OBS
    with open(path, "w") as f:
        json.dump(r, f, indent=1)
    dev = REF["NQS"][L][0] - r["E"]
    print(f"L={L}: E = {r['E']:.6f} +- {r['E_err']:.6f}  chi2_red={r['chi2_red']:.2f}  "
          f"[{(time.time()-t0)/60:.1f} min]   E_NQS - E_QMC = {dev:+.4f}")

In [ ]:
# ====== 6 · PMRQMC cross-check at L=4 (independent algorithm) ======
# H.txt generator: verbatim copy of analysis/scripts/export_pmrqmc.py::cubic_obc_stabilizers,
# verified ground-state-isomorphic to ThreeD_ToricCodeGeometry at L=2 OBC (1e-9).
def cubic_obc_stabilizers(L):
    def vid(x, y, z):
        return x + L * y + L * L * z
    edge_id = {}
    for z in range(L):
        for y in range(L):
            for x in range(L):
                if x < L - 1: edge_id[(vid(x, y, z), 0)] = len(edge_id)
                if y < L - 1: edge_id[(vid(x, y, z), 1)] = len(edge_id)
                if z < L - 1: edge_id[(vid(x, y, z), 2)] = len(edge_id)
    steps = ((1, 0, 0), (0, 1, 0), (0, 0, 1))
    stars = []
    for z in range(L):
        for y in range(L):
            for x in range(L):
                star = []
                for d, (dx, dy, dz) in enumerate(steps):
                    e = edge_id.get((vid(x, y, z), d))
                    if e is not None: star.append(e)
                    e = edge_id.get((vid(x - dx, y - dy, z - dz), d))
                    if e is not None: star.append(e)
                stars.append(star)
    plaqs = []
    for z in range(L):
        for y in range(L):
            for x in range(L):
                v = vid(x, y, z)
                for da, db in ((0, 1), (0, 2), (1, 2)):
                    sa, sb = steps[da], steps[db]
                    if (x, y, z)[da] < L - 1 and (x, y, z)[db] < L - 1:
                        va = vid(x + sa[0], y + sa[1], z + sa[2])
                        vb = vid(x + sb[0], y + sb[1], z + sb[2])
                        plaqs.append([edge_id[(v, da)], edge_id[(v, db)],
                                      edge_id[(va, db)], edge_id[(vb, da)]])
    return len(edge_id), stars, plaqs

if RUN_PMRQMC:
    L = 4
    path = f"{OUTDIR}/raw/pmrqmc_L{L}_hx{HX}_hz{HZ}_beta{PMR_BETA}.json"
    if os.path.exists(path):
        print(f"PMRQMC L={L}: exists, skipping")
    else:
        wd = "/content/pmrqmc_run"
        subprocess.run(["bash", "-c", f'''
            set -e
            cd /content
            [ -d PMRQMC ] || git clone --depth 1 https://github.com/LevBarash/PMRQMC
            mkdir -p {wd} && cp PMRQMC/prepare.cpp PMRQMC/PMRQMC.cpp PMRQMC/mainQMC.hpp \
              PMRQMC/divdiff.hpp PMRQMC/parameters.hpp {wd}/'''], check=True)
        N, stars, plaqs = cubic_obc_stabilizers(L)
        lines = ["-1 " + " ".join(f"{e+1} X" for e in sorted(s)) for s in stars]
        lines += ["-1 " + " ".join(f"{e+1} Z" for e in sorted(p)) for p in plaqs]
        for h, s in ((HX, "X"), (HZ, "Z")):
            if h != 0.0:
                lines += [f"{-h} {e+1} {s}" for e in range(N)]
        with open(f"{wd}/H.txt", "w") as f:
            f.write("\n".join(lines) + "\n")
        pp = open(f"{wd}/parameters.hpp").read()
        for pat, sub in ((r"#define beta\s+\S+",   f"#define beta   {PMR_BETA}"),
                         (r"#define qmax\s+\S+",   f"#define qmax     {PMR_QMAX}"),
                         (r"#define Tsteps\s+\S+", f"#define Tsteps {PMR_TSTEPS}"),
                         (r"#define steps\s+\S+",  f"#define steps  {PMR_STEPS}")):
            pp = re.sub(pat, sub, pp, count=1)
        open(f"{wd}/parameters.hpp", "w").write(pp)
        subprocess.run(["bash", "-c", f'''
            set -e
            cd {wd}
            g++ -O3 -std=c++11 -o prepare.bin prepare.cpp && ./prepare.bin H.txt
            g++ -O3 -std=c++11 -o PMRQMC.bin PMRQMC.cpp'''], check=True)
        procs = []
        for i in range(PMR_CHAINS):
            d = f"{wd}/chain{i}"
            os.makedirs(d, exist_ok=True)
            for fn in ("PMRQMC.bin", "hamiltonian.hpp"):
                subprocess.run(["cp", f"{wd}/{fn}", d], check=True)
            procs.append(subprocess.Popen([f"{d}/PMRQMC.bin"], cwd=d,
                                          stdout=subprocess.PIPE, text=True))
        chains = []
        for p in procs:
            out, _ = p.communicate()
            assert "Warning: qmax" not in out, "raise PMR_QMAX and rerun"
            m = re.search(r"Observable #\d+: H\s*\nmean\(O\) = (\S+)\s*\nstd\.dev\.\(O\) = (\S+)", out)
            q = re.search(r"mean\(q\) = (\S+)\s*\nmax\(q\) = (\S+)", out)
            chains.append(dict(E=float(m.group(1)), E_err=float(m.group(2)),
                               mean_q=float(q.group(1)), max_q=float(q.group(2))))
            print(f"chain: E = {chains[-1]['E']:.6f} +- {chains[-1]['E_err']:.6f}  "
                  f"<q>={chains[-1]['mean_q']:.0f} max(q)={chains[-1]['max_q']:.0f}")
        E = np.array([c["E"] for c in chains]); sE = np.array([c["E_err"] for c in chains])
        w = 1.0 / sE**2
        r = dict(engine="pmrqmc", L=L, hx=HX, hz=HZ, beta=PMR_BETA,
                 E=float(np.sum(w * E) / np.sum(w)), E_err=float(np.sqrt(1.0 / np.sum(w))),
                 chains=chains, steps=PMR_STEPS)
        with open(path, "w") as f:
            json.dump(r, f, indent=1)
        print(f"PMRQMC L={L}: E = {r['E']:.6f} +- {r['E_err']:.6f}")

## 7 · Comparison

$E_{\rm NQS}$ is variational: the truth satisfies $E_0\le E_{\rm NQS}$, and QMC estimates $E_0$
without bias (thermal offset $<10^{-7}$ at $\beta=12$, checked by the $\beta$-doubling row).
So $\Delta=E_{\rm NQS}-E_{\rm QMC}$ **is** the NQS variational error $\varepsilon$, measured with
uncertainty $\sigma=\sqrt{\sigma_{\rm QMC}^2+\sigma_{\rm NQS}^2}$:
$\Delta$ consistent with 0 at $2\sigma$ → only the bound $\varepsilon<\Delta+2\sigma$;
$\Delta>3\sigma$ → a *measured* variational error (expected at $L=6,7$ from the pure-$h_z$
certificate); $\Delta<-3\sigma$ → impossible for a variational state — a bug in one of the codes.
The 2nd-order series column shows what this measurement adds: at a mixed point its $O(h^4)$
truncation is unknown, so the series alone cannot separate truncation from variational error.

In [ ]:
# ====== 8 · Comparison table + figure ======
import glob as _glob
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

rows = []
for L in LS:
    p = f"{OUTDIR}/raw/paratoric_L{L}_hx{HX}_hz{HZ}_beta{BETA}.json"
    if not os.path.exists(p):
        continue
    r = json.load(open(p))
    En, sn = REF["NQS"][L]
    d, s = En - r["E"], float(np.hypot(sn, r["E_err"]))
    rows.append(dict(L=L, engine="ParaToric", Eq=r["E"], sq=r["E_err"], En=En, sn=sn,
                     dev=d, sig=s, z=d / s, ser2=series2(L, HX, HZ)))
for p in _glob.glob(f"{OUTDIR}/raw/pmrqmc_L*_hx{HX}_hz{HZ}_*.json"):
    r = json.load(open(p))
    En, sn = REF["NQS"][r["L"]]
    d, s = En - r["E"], float(np.hypot(sn, r["E_err"]))
    rows.append(dict(L=r["L"], engine="PMRQMC", Eq=r["E"], sq=r["E_err"], En=En, sn=sn,
                     dev=d, sig=s, z=d / s, ser2=series2(r["L"], HX, HZ)))

print(f"{'L':>2s} {'engine':>9s} {'E_QMC':>13s} {'err':>9s} {'E_NQS':>13s} {'err':>7s} "
      f"{'eps=E_NQS-E_QMC':>16s} {'z':>6s}  verdict")
for r in sorted(rows, key=lambda r: (r["L"], r["engine"])):
    v = ("NQS error RESOLVED" if r["z"] > 3 else
         "IMPOSSIBLE (bug)"  if r["z"] < -3 else
         f"agree; eps < {r['dev'] + 2*r['sig']:.4f} (2sig)")
    print(f"{r['L']:2d} {r['engine']:>9s} {r['Eq']:13.4f} {r['sq']:9.4f} {r['En']:13.4f} "
          f"{r['sn']:7.4f} {r['dev']:+16.4f} {r['z']:6.1f}  {v}")

pt = [r for r in rows if r["engine"] == "ParaToric"]
if pt:
    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    col = dict(zip(LS, plt.cm.plasma(np.linspace(0, 0.85, len(LS)))))
    for r in pt:   # everything relative to the exact 2nd-order series at this point
        ax.errorbar(r["L"] - 0.06, r["Eq"] - r["ser2"], yerr=r["sq"], fmt="o",
                    color=col[r["L"]], ms=6, capsize=3)
        ax.errorbar(r["L"] + 0.06, r["En"] - r["ser2"], yerr=r["sn"], fmt="s",
                    mfc="white", color=col[r["L"]], ms=6, capsize=3)
    ax.axhline(0.0, color="0.6", lw=0.8, ls="--")
    ax.set_xlabel("$L$"); ax.set_xticks(LS)
    ax.set_ylabel(r"$E - E_{\rm series}^{(2)}$")
    ax.set_title(rf"$(h_x,h_z)=({HX},{HZ})$: QMC (filled) vs NQS (open); "
                 r"dashed $=$ 2nd-order series", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/qmc_vs_nqs_hx{HX}_hz{HZ}.png", dpi=300, bbox_inches="tight")
    plt.show()
print(f"figure + JSONs in {OUTDIR}")